[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C54_DETR_Set_Prediction_Course/03_queries/03_object_queries.ipynb)

# 03 · Object query 与交叉注意力（注意力实现 / 职责分工 / 空间特化 / DAB 调制）

目标：把 DETR decoder **从零手写一遍**，并用可证伪的实验回答三个问题——
**query 是什么？self-attention 和 cross-attention 各干什么？query 训练完变成了什么？**

**本 notebook 你会亲手实现：**
1. `attention` / `mha` —— 单头与多头缩放点积注意力（numpy），并证明 `h=1` 时二者等价
2. **decoder 一层**：self-attn + cross-attn + FFN，位置编码只加 Q/K 不加 V
3. **关键扰动实验**：拿掉 self-attention 后，改动 query 5 对其余 query 的输出
   **逐比特没有影响** —— 精确证明「self-attention 是 query 之间唯一的通道」
4. **去重实验**：静态输出 vs 输入条件抑制，量化重复率与召回的变化
5. **query 空间特化**：用匈牙利匹配 + 梯度下降真训练一批 query 先验，
   看它们如何自发分工；再换成 **TSR 风格的偏置分布**，复现「query 特化 = 数据分布的镜像」
6. **DAB-DETR 的宽高调制**：证明正弦位置编码的点积是一个钟形核，
   且把坐标除以 `w` 会精确地把核**拉宽 w 倍**
7. 四道练习：Conditional 分解 / 逐层框精修 / N 的选择 / 注意力有效范围

> 心智模型：**一对一匹配提供「不许重复」的压力，self-attention 提供执行这个压力的通道。
> 缺任何一个，DETR 都得重新装回 NMS。**

## 1 · 注意力从零实现：单头 → 多头

先把工具造出来。两个必须验证的性质：**每行权重和为 1**、**mask 掉的 key 权重为 0**。

In [ ]:
import numpy as np, math, itertools
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def attention(Q, K, V, mask=None):
    '''缩放点积注意力。Q (nq,d), K (nk,d), V (nk,dv) -> (out (nq,dv), A (nq,nk))'''
    d = Q.shape[-1]
    logits = Q @ K.T / np.sqrt(d)              # 除以 sqrt(d)：防止 d 大时 softmax 饱和
    if mask is not None:
        logits = np.where(mask, logits, -1e9)  # mask=False 的位置被压到 -inf
    A = softmax(logits, -1)
    return A @ V, A

d, nq, nk = 16, 5, 9
Q = rng.normal(size=(nq, d)); K = rng.normal(size=(nk, d)); V = rng.normal(size=(nk, d))
out, A = attention(Q, K, V)
print('out', out.shape, '| A', A.shape)
assert np.allclose(A.sum(-1), 1.0), '每行注意力权重必须和为 1'
assert (A >= 0).all()

# mask：被屏蔽的 key 权重必须是 0（DN-DETR 的去噪分组隔离就靠它，见模块 04）
mask = np.ones((nq, nk), dtype=bool); mask[:, 3:6] = False
_, Am = attention(Q, K, V, mask)
assert Am[:, 3:6].max() < 1e-9, 'mask 掉的 key 权重必须≈0'
assert np.allclose(Am.sum(-1), 1.0)
print('mask 后被屏蔽列的最大权重 = %.2e  ✅' % Am[:, 3:6].max())

# 温度效应：logits 放大 -> 注意力从「弥散」走向「聚焦」
print('\n%-10s %14s %14s' % ('logits 缩放', '最大权重', '有效关注范围'))
for t in [0.0, 0.5, 1.0, 4.0, 16.0]:
    _, At = attention(Q * t, K, V)
    H = -(At * np.log(At + 1e-12)).sum(-1)
    print('%-10.1f %14.4f %14.2f' % (t, At.max(1).mean(), np.exp(H).mean()))
print('\n⚠️  t=0（等价于训练刚初始化）时有效关注范围 = %d = 全部 key —— ' % nk)
print('    **每个 key 只分到 1/%d 的梯度**。这就是 DETR 收敛慢的第一个根因（第 3 节展开）。' % nk)

In [ ]:
def mha(Xq, Xk, Xv, W, h):
    '''多头注意力。W = {q,k,v,o} 四个 (d,d) 投影矩阵。返回 (out (nq,d), A (h,nq,nk))'''
    nq, dm = Xq.shape
    nk = Xk.shape[0]
    dh = dm // h
    assert dm % h == 0, 'd_model 必须能被头数整除'
    Qh = (Xq @ W['q']).reshape(nq, h, dh).transpose(1, 0, 2)   # (h, nq, dh)
    Kh = (Xk @ W['k']).reshape(nk, h, dh).transpose(1, 0, 2)
    Vh = (Xv @ W['v']).reshape(nk, h, dh).transpose(1, 0, 2)
    logits = np.einsum('hid,hjd->hij', Qh, Kh) / np.sqrt(dh)   # 注意是除 sqrt(dh) 不是 sqrt(d)
    A = softmax(logits, -1)
    O = np.einsum('hij,hjd->hid', A, Vh).transpose(1, 0, 2).reshape(nq, dm)
    return O @ W['o'], A

W = {k: rng.normal(0, 0.5, (d, d)) for k in 'qkvo'}
Xq = rng.normal(size=(nq, d)); Xk = rng.normal(size=(nk, d))

# h=1 时多头必须退化成「先投影再单头注意力再输出投影」
o1, A1 = mha(Xq, Xk, Xk, W, h=1)
o_ref, A_ref = attention(Xq @ W['q'], Xk @ W['k'], Xk @ W['v'])
assert np.allclose(o1, o_ref @ W['o']), 'h=1 必须与单头等价'
assert np.allclose(A1[0], A_ref)
print('✅ h=1 与单头实现逐元素一致（最大差 %.2e）' % np.abs(o1 - o_ref @ W['o']).max())

# 多头：不同 head 会关注不同的 key —— 这正是 DETR 里「不同 head 负责框的不同边」的机制基础
o4, A4 = mha(Xq, Xk, Xk, W, h=4)
assert A4.shape == (4, nq, nk) and np.allclose(A4.sum(-1), 1.0)
print('\n每个 head 对 query#0 的 top-1 key:',
      [int(A4[hh, 0].argmax()) for hh in range(4)])
n_distinct = len({int(A4[hh, 0].argmax()) for hh in range(4)})
print('4 个 head 关注了 %d 个不同的 key  ← **多头 = 多个并行的关注模式**' % n_distinct)
assert o4.shape == (nq, d)

## 2 · Decoder 一层：self-attn + cross-attn + FFN

严格按 DETR 官方实现的连接方式（post-LN）：

- `tgt` 初始化为**全 0**（内容），`query_pos` 是**可学习位置嵌入**
- **位置编码只加到 Q 和 K，绝不加到 V** —— 位置决定「往哪看」，不决定「取回什么」

In [ ]:
def layer_norm(x, eps=1e-5):
    return (x - x.mean(-1, keepdims=True)) / np.sqrt(x.var(-1, keepdims=True) + eps)

def ffn(x, W1, b1, W2, b2):
    return np.maximum(x @ W1 + b1, 0.0) @ W2 + b2

def make_params(d=32, dff=64, h=4, seed=0):
    r = np.random.default_rng(seed)
    def proj():
        return {k: r.normal(0, 1 / np.sqrt(d), (d, d)) for k in 'qkvo'}
    return {'h': h, 'self': proj(), 'cross': proj(),
            'ffn': (r.normal(0, 1 / np.sqrt(d), (d, dff)), np.zeros(dff),
                    r.normal(0, 1 / np.sqrt(dff), (dff, d)), np.zeros(d))}

def decoder_layer(tgt, query_pos, memory, mem_pos, Pm, use_self_attn=True):
    '''返回 (tgt_out, A_self, A_cross)。'''
    A_self = None
    if use_self_attn:
        q = k = tgt + query_pos                                  # pos 进 Q/K
        sa, A_self = mha(q, k, tgt, Pm['self'], Pm['h'])         # **V = tgt，不加 pos**
        tgt = layer_norm(tgt + sa)
    ca, A_cross = mha(tgt + query_pos,                            # Q：query + 位置先验
                      memory + mem_pos,                           # K：图像特征 + 2D 位置编码
                      memory,                                     # **V：纯图像特征**
                      Pm['cross'], Pm['h'])
    tgt = layer_norm(tgt + ca)
    tgt = layer_norm(tgt + ffn(tgt, *Pm['ffn']))
    return tgt, A_self, A_cross

D, H, NQ, HW = 32, 4, 8, 40
Pm = make_params(D, 64, H)
memory   = rng.normal(0, 1, (HW, D))          # encoder 输出（图像证据）
mem_pos  = rng.normal(0, 0.5, (HW, D))        # 2D sine 位置编码（这里用随机向量替代）
query_pos = rng.normal(0, 1, (NQ, D))         # **可学习的 object query**
tgt0 = np.zeros((NQ, D))                      # **内容初始化为全 0**

out1, A_self_demo, A_cross_demo = decoder_layer(tgt0, query_pos, memory, mem_pos, Pm)
print('tgt  ', tgt0.shape, ' -> ', out1.shape)
print('A_self ', A_self_demo.shape, '  (h, N, N)      ← query 之间')
print('A_cross', A_cross_demo.shape, '  (h, N, HW)     ← query -> 图像')
assert np.allclose(A_cross_demo.sum(-1), 1.0) and np.allclose(A_self_demo.sum(-1), 1.0)

# 初始化时 cross-attention 有多「弥散」？
Hc = -(A_cross_demo * np.log(A_cross_demo + 1e-12)).sum(-1)
span0 = float(np.exp(Hc).mean())
print('\n初始化时 cross-attention 的有效关注范围 = %.1f / %d 个 key（%.0f%%）'
      % (span0, HW, 100 * span0 / HW))
assert span0 > 0.4 * HW, '随机初始化时注意力应远未聚焦'
print('⚠️  真实 DETR 的 HW ≈ 850–1000，初始 span 同样占全部 key 的一半以上 ——')
print('    **每个 key 只分到几百分之一的梯度**，模型要先花几百个 epoch 把注意力收窄。')
print('    Deformable DETR 让每个 query 只采样 K=4 个点，就是直接根治这一条（模块 04）。')

# tgt 全 0 时，第一层 self-attention 的 V 也全 0 -> 输出全 0 -> **第一层 self-attn 几乎无用**
sa_out = mha(tgt0 + query_pos, tgt0 + query_pos, tgt0, Pm['self'], H)[0]
assert np.abs(sa_out).max() < 1e-12
print('\n✅ 第一层 self-attention 的输出恒为 0（因为 V = tgt = 0）——')
print('   这正好解释了 DETR 论文的消融：**去掉第一层 self-attn，AP 几乎不掉**。')

## 3 · 关键实验：只有 self-attention 能让 query 之间互相看见

**可证伪的断言**：cross-attention 与 FFN 都是逐 query 独立（pointwise）的运算。
所以拿掉 self-attention 后，改动 query 5 的位置嵌入，
**其余 query 的输出必须逐比特不变**。

In [ ]:
# 第一层 tgt=0 时 self-attn 的 V 也是 0（上一节已证），所以这里模拟**第 2 层及以后**：
# tgt 已经通过 cross-attention 积累了内容。tgt_mid 在整个实验里保持不变。
tgt_mid = rng.normal(0, 1, (NQ, D))

def probe(use_self_attn, victim=5, scale=3.0):
    '''扰动 query[victim] 的位置嵌入，测量其余 query 输出的变化幅度。'''
    base, _, _ = decoder_layer(tgt_mid, query_pos, memory, mem_pos, Pm, use_self_attn)
    qp2 = query_pos.copy()
    qp2[victim] = qp2[victim] + scale * rng2.normal(0, 1, D)
    pert, _, _ = decoder_layer(tgt_mid, qp2, memory, mem_pos, Pm, use_self_attn)
    others = [i for i in range(NQ) if i != victim]
    return (float(np.abs(pert[others] - base[others]).max()),
            float(np.abs(pert[victim] - base[victim]).max()))

rng2 = np.random.default_rng(7)
d_other_no, d_self_no = probe(use_self_attn=False)
rng2 = np.random.default_rng(7)                      # 同一个扰动，保证可比
d_other_sa, d_self_sa = probe(use_self_attn=True)

print('%-26s %22s %20s' % ('decoder 配置', '其余 query 输出变化', '被扰动 query 自己'))
print('%-24s %22.3e %20.4f' % ('❌ 无 self-attention', d_other_no, d_self_no))
print('%-24s %22.3e %20.4f' % ('✅ 有 self-attention', d_other_sa, d_self_sa))

assert d_other_no < 1e-12, '无 self-attn 时，其余 query 的输出必须**逐比特不变**'
assert d_self_no > 1e-3,   '被扰动的 query 自己当然会变'
assert d_other_sa > 1e-3,  '有 self-attn 时，其余 query 必须受影响'
print('\n✅ **数学证明级别的结论**：')
print('   拿掉 self-attention 后，decoder 就是 %d 条互不相干的流水线 ——' % NQ)
print('   query 之间**没有任何通信通道**，于是两个偏好相近的 query 会输出几乎相同的框，')
print('   而且谁也没办法「知道对方已经认领了」从而退让。→ **重复框，必须外挂 NMS**。')
print('\n⚠️  面试标准答案：一对一匹配提供「不许重复」的**压力**（模块 02），')
print('    self-attention 提供执行这个压力的**通道**（本节）。**缺一不可。**')

## 4 · 去重：静态输出 vs 输入条件抑制

上一节证明了「通道」的必要性。这一节量化「有通道」能带来什么：
把 self-attention 学到的行为写成一个显式的**输入条件抑制**（被更自信的邻居压制），
对比它与「各自为战」的重复率和召回。

> 注意：下面的抑制函数是**手写的替身**，用来展示 self-attention *必须学会做什么*；
> 真实 DETR 里这个行为是从一对一匹配的梯度里学出来的，且判据可以远比距离丰富。

In [ ]:
def make_scene(rg, n_query=20):
    '''合成一帧：m 个目标，每个被 2-3 个 query 认领（=重复），其余 query 输出低分背景。'''
    m = int(rg.integers(2, 5))
    gts = rg.uniform(0.10, 0.90, size=(m, 2))
    centers, scores, owner = [], [], []
    for j in range(m):
        for _ in range(int(rg.integers(2, 4))):
            centers.append(gts[j] + rg.normal(0, 0.008, 2))
            scores.append(float(rg.uniform(0.45, 0.95)))
            owner.append(j)
    while len(centers) < n_query:
        centers.append(rg.uniform(0, 1, 2)); scores.append(float(rg.uniform(0.0, 0.25)))
        owner.append(-1)
    return gts, np.array(centers), np.array(scores), np.array(owner)

def self_attn_inhibition(centers, scores, lam=1.0, sigma=0.03):
    '''self-attention 学到的行为：**只被比自己更自信的邻居抑制**（不对称）。'''
    D2 = ((centers[:, None, :] - centers[None, :, :]) ** 2).sum(-1)
    Wk = np.exp(-D2 / (2 * sigma ** 2))          # 距离核（真实模型里是学出来的相似度）
    np.fill_diagonal(Wk, 0.0)
    stronger = (scores[None, :] > scores[:, None]).astype(float)
    inhib = (Wk * stronger * scores[None, :]).max(1)
    return scores - lam * inhib

def evaluate(rg, use_inhibition, n_scene=400, thr=0.30):
    dup, rec = [], []
    for _ in range(n_scene):
        gts, ctr, sc, own = make_scene(rg)
        s = self_attn_inhibition(ctr, sc) if use_inhibition else sc
        keep = s > thr
        fg = own[keep][own[keep] >= 0]
        dup.append(len(fg) - len(set(fg.tolist())))          # 重复框数
        rec.append(len(set(fg.tolist())) / len(gts))         # 召回
    return float(np.mean(dup)), float(np.mean(rec))

dup_no, rec_no = evaluate(np.random.default_rng(3), False)
dup_sa, rec_sa = evaluate(np.random.default_rng(3), True)
print('%-30s %14s %12s' % ('', '每帧重复框数', '召回'))
print('%-28s %14.3f %12.3f' % ('❌ 各自为战（无通信）', dup_no, rec_no))
print('%-28s %14.3f %12.3f' % ('✅ 输入条件抑制（self-attn）', dup_sa, rec_sa))
assert dup_no > 2.0, '没有通信时应有大量重复'
assert dup_sa < 0.2 * dup_no, '抑制后重复框应下降一个数量级'
assert rec_sa > 0.90 * rec_no, '召回不应被显著牺牲'
print('\n✅ 重复框 %.2f -> %.2f（降低 %.0f%%），召回 %.3f -> %.3f（几乎不变）'
      % (dup_no, dup_sa, 100 * (1 - dup_sa / dup_no), rec_no, rec_sa))
print('\n⚠️  与 NMS 的关键差别：NMS 的判据**只有 IoU**，且阈值是手调的全局常数。')
print('    self-attention 的判据可以是外观 / 语义 / 上下文 —— 所以它能学会')
print('    「同一根立杆上下两块牌虽然框高度重叠，但是两个物体」，')
print('    而 IoU 阈值在这种 TSR 场景里怎么设都是错的（设高放重复，设低删真目标）。')

## 5 · Query 的空间特化：用匈牙利匹配真训练一遍

**没有任何损失项要求 query 分工**，但一对一匹配会让它们自发分开：
两个偏好相同的 query 会永远争夺同一批 GT，其中一个必然拿到 ∅ 标签 —— 分开才是损失更低的解。

下面用一个极小的可训练模型复现这个现象：每个 query 只有一个 2D 位置先验 `p_i`，
损失是「匹配上的 (query, GT) 的 L2 距离平方」，用匈牙利匹配 + 梯度下降训练。

In [ ]:
def hungarian(cost):
    '''O(n^3) 匈牙利算法（模块 01/02 已实现，这里直接复用）。要求 n <= m。'''
    C = np.asarray(cost, dtype=float)
    n, m = C.shape
    assert n <= m
    INF = float('inf')
    u = np.zeros(n + 1); v = np.zeros(m + 1)
    p = np.zeros(m + 1, dtype=int); way = np.zeros(m + 1, dtype=int)
    for i in range(1, n + 1):
        p[0] = i; j0 = 0
        minv = np.full(m + 1, INF); used = np.zeros(m + 1, dtype=bool)
        while True:
            used[j0] = True
            i0 = p[j0]; delta = INF; j1 = -1
            for j in range(1, m + 1):
                if not used[j]:
                    cur = C[i0 - 1, j - 1] - u[i0] - v[j]
                    if cur < minv[j]:
                        minv[j] = cur; way[j] = j0
                    if minv[j] < delta:
                        delta = minv[j]; j1 = j
            for j in range(m + 1):
                if used[j]:
                    u[p[j]] += delta; v[j] -= delta
                else:
                    minv[j] -= delta
            j0 = j1
            if p[j0] == 0:
                break
        while j0:
            j1 = way[j0]; p[j0] = p[j1]; j0 = j1
    col = np.zeros(n, dtype=int)
    for j in range(1, m + 1):
        if p[j]:
            col[p[j] - 1] = j - 1
    return np.arange(n), col

def scene_uniform(rg):
    '''目标均匀散布在整张图上。'''
    return rg.uniform(0.05, 0.95, size=(int(rg.integers(2, 7)), 2))

def scene_tsr(rg):
    '''**TSR 风格的偏置分布**：竖直方向集中在地平线上方一条窄带，
       水平方向 75% 落在道路右侧（右行 + 立杆在右）。'''
    m = int(rg.integers(1, 5))
    y = rg.normal(0.30, 0.05, m)
    right = rg.random(m) < 0.75
    x = np.where(right, rg.normal(0.80, 0.08, m), rg.normal(0.22, 0.08, m))
    return np.clip(np.stack([x, y], -1), 0.02, 0.98)

def train_query_priors(scene_fn, n_query=12, steps=800, lr=0.08, seed=0):
    '''每个 query 一个 2D 位置先验；一对一匹配 + SGD。'''
    rg = np.random.default_rng(seed)
    Pq = rg.uniform(0.45, 0.55, size=(n_query, 2))      # 初始：**全挤在画面中央**
    hits = np.zeros(n_query)
    for _ in range(steps):
        gts = scene_fn(rg)
        C = ((Pq[:, None, :] - gts[None, :, :]) ** 2).sum(-1)     # (N, M)
        r, c = hungarian(C.T)                                      # 行=GT，列=query
        for gi, qi in zip(r, c):
            Pq[qi] -= lr * 2.0 * (Pq[qi] - gts[gi])                # d/dp ||p-g||^2
            hits[qi] += 1
        Pq = np.clip(Pq, 0.02, 0.98)
    return Pq, hits

def coverage(Pq, scene_fn, seed=99, n=400):
    '''平均「GT 到最近 query 先验」的距离 —— 越小说明 query 覆盖越好。'''
    rg = np.random.default_rng(seed); tot = []
    for _ in range(n):
        g = scene_fn(rg)
        tot.append(np.sqrt(((g[:, None, :] - Pq[None, :, :]) ** 2).sum(-1)).min(1).mean())
    return float(np.mean(tot))

P_init = np.random.default_rng(0).uniform(0.45, 0.55, size=(12, 2))
P_uni, hits_uni = train_query_priors(scene_uniform)
c0, c1 = coverage(P_init, scene_uniform), coverage(P_uni, scene_uniform)
print('均匀分布数据：覆盖距离 %.4f（训练前，全挤在中央） -> %.4f（训练后）' % (c0, c1))
assert c1 < 0.6 * c0, '训练后 query 应显著铺开'
print('\n训练后 12 个 query 的位置先验（x, y）与被认领次数：')
for i in np.argsort(-hits_uni):
    print('  query %2d  (%.3f, %.3f)   认领 %4d 次' % (i, P_uni[i, 0], P_uni[i, 1], hits_uni[i]))
spread = P_uni.std(0)
assert (spread > 0.15).all(), 'query 在两个方向上都应该铺开'
print('\n✅ **没有任何损失项要求它们分工，但它们自发分开了**（std = %.3f, %.3f）。'
      % (spread[0], spread[1]))
print('   机理：两个偏好相同的 query 会永远争同一批 GT，输的那个拿 ∅ 标签 —— 分开才更优。')

In [ ]:
# 换成 **TSR 风格的偏置分布**：看 query 特化如何变成「数据分布的镜像」
P_tsr, hits_tsr = train_query_priors(scene_tsr, seed=1)
active = hits_tsr >= 20
print('被有效训练到的 query: %d / %d' % (active.sum(), len(hits_tsr)))
print('\n%-10s %10s %10s %10s' % ('query', 'x（左右）', 'y（上下）', '认领次数'))
for i in np.argsort(-hits_tsr):
    tag = ''
    if hits_tsr[i] >= 20:
        tag = '  ← 右侧带' if P_tsr[i, 0] > 0.5 else '  ← 左侧带'
    print('%-10d %10.3f %10.3f %10d%s' % (i, P_tsr[i, 0], P_tsr[i, 1], hits_tsr[i], tag))

ya = P_tsr[active, 1]; xa = P_tsr[active, 0]
print('\n有效 query 的 y 均值 = %.3f（数据集中在 0.30 附近）' % ya.mean())
print('有效 query 中落在右半边的比例 = %.2f（数据里 75%% 的标志在右侧）'
      % (xa > 0.5).mean())
assert ya.mean() < 0.45, 'query 应集中到「地平线上方」那条窄带'
assert (xa > 0.5).mean() >= 0.5, '多数 query 应特化到道路右侧'
cov_tsr = coverage(P_tsr, scene_tsr)
print('TSR 分布上的覆盖距离 = %.4f' % cov_tsr)

# **关键的失效实验**：把这批 query 直接拿去跑「左行国家 / 城市道路」分布
def scene_shifted(rg):
    '''分布外场景：标志出现在画面左侧、且位置更低（城市道路 / 左行）。'''
    m = int(rg.integers(1, 5))
    y = rg.normal(0.55, 0.08, m)
    x = np.where(rg.random(m) < 0.75, rg.normal(0.22, 0.08, m), rg.normal(0.80, 0.08, m))
    return np.clip(np.stack([x, y], -1), 0.02, 0.98)

cov_ood = coverage(P_tsr, scene_shifted)
print('\n分布外（左行 / 城市道路）上的覆盖距离 = %.4f  ← 恶化 %.1f 倍'
      % (cov_ood, cov_ood / cov_tsr))
assert cov_ood > 2.0 * cov_tsr, '分布外场景上 query 覆盖应显著变差'
print('\n⚠️  **这是一个离线 mAP 看不出、实车上会成片掉召回的失效模式**：')
print('    query 的特化是数据分布的镜像；没被训练覆盖的区域没有任何 query 负责。')
print('✅ 对策：① 评测按「标志在画面中的位置」分桶（C55 模块 05）；')
print('        ② 数据要覆盖各类道路几何（匝道 / 环岛 / 城市 / 龙门架）；')
print('        ③ **慎用水平翻转增强** —— 它能补左侧分布，但会把「左转」翻成「右转」、')
print('           把牌面文字镜像（C56 模块 01 的语义禁区）。主动指出这一点是面试加分项。')

## 6 · DAB-DETR：为什么「除以 w」就等于把注意力窗口拉宽 w 倍

正弦位置编码有一个漂亮的性质：**两点位置编码的点积只依赖于它们的距离**，
并且是一个以 0 为峰的钟形核：

$$\mathrm{PE}(x)^\top \mathrm{PE}(x') = \sum_j \cos\big(\omega_j (x-x')\big)$$

所以把坐标除以 `w`，就等于把这个核在空间上**精确地拉宽 w 倍** —— 这就是 DAB-DETR
用 anchor 的宽高调制位置注意力的全部原理。

In [ ]:
def sinusoidal_pe(x, d=128, temperature=20.0):
    '''DETR 风格的 1D 正弦位置编码。x: (n,) -> (n, d)'''
    i = np.arange(d // 2)
    omega = 1.0 / (temperature ** (2 * i / d))
    ang = x[:, None] * omega[None, :] * 2 * np.pi
    return np.concatenate([np.sin(ang), np.cos(ang)], -1)

# 性质 1：点积只依赖距离（平移不变）
def pe_dot(x, y):
    return float(sinusoidal_pe(np.array([x]))[0] @ sinusoidal_pe(np.array([y]))[0])

a, b = 0.31, 0.77
lhs, rhs = pe_dot(a, b), pe_dot(0.0, b - a)
assert abs(lhs - rhs) < 1e-9, 'PE(a)·PE(b) 必须只依赖 b-a'
print('PE(%.2f)·PE(%.2f) = %.4f ;  PE(0)·PE(%.2f) = %.4f   ✅ 平移不变'
      % (a, b, lhs, b - a, rhs))

def pe_kernel(deltas, w=1.0, d=128, T=20.0):
    '''把坐标除以 w 之后的位置注意力核 k(delta)。'''
    K0 = sinusoidal_pe(np.array([0.0]) / w, d, T)[0]
    return sinusoidal_pe(deltas / w, d, T) @ K0

def half_width(deltas, ker):
    '''核降到峰值一半时的 delta —— 即「注意力窗口的半宽」。'''
    below = np.where(ker < ker[0] / 2.0)[0]
    return float(deltas[below[0]]) if len(below) else float('nan')

deltas = np.linspace(0, 1.2, 2401)
print('\n%-14s %12s %14s' % ('anchor 宽度 w', '核峰值', '注意力半宽'))
hw = {}
for w in [0.25, 0.5, 1.0, 2.0]:
    ker = pe_kernel(deltas, w)
    hw[w] = half_width(deltas, ker)
    print('%-14.2f %12.1f %14.4f' % (w, ker[0], hw[w]))

for w in [0.25, 0.5, 2.0]:
    ratio = hw[w] / hw[1.0]
    assert abs(ratio - w) < 0.05 * w, 'w=%.2f 时半宽比应≈%.2f，实得 %.3f' % (w, w, ratio)
print('\n✅ 半宽与 w **精确成正比**（误差 < 5%）——')
print('   宽目标得到宽窗口，8x8 的远处限速牌得到窄窗口。')

# 画一下核的形状（ASCII）
print('\n位置注意力核 k(delta) 的形状（w=0.5 窄 vs w=2.0 宽）：')
for w, mark in [(0.5, '#'), (2.0, '=')]:
    ker = pe_kernel(deltas, w)
    line = ''.join(mark if ker[int(t / 1.2 * 2400)] > ker[0] / 2 else '.'
                   for t in np.linspace(0, 1.19, 60))
    print('  w=%.1f |%s|  半宽=%.3f' % (w, line, hw[w]))
print('        0%s1.2   (delta)' % (' ' * 57))
print('\n⚠️  Conditional DETR 的空间项是**各向同性**的固定圆斑；')
print('    DAB-DETR 用 (w, h) 分别调制 x / y 两个方向 —— 于是窗口能变成扁的、长的、小的。')
print('✅ 顺带的第二个红利：既然 query 就是一个 (x,y,w,h) 的框，它就能被**逐层精修**')
print('   （练习 2），这与模块 02 的辅助损失是天生一对。')

## 7 · Query 数量 N 的账：覆盖率 / 正样本比例 / 计算量

In [ ]:
counts_coco = rng.poisson(7.7, 40000)      # COCO 风格：单图平均 7.7 个实例
counts_tsr  = rng.poisson(1.8, 40000)      # **TSR 风格：一张高速前视图通常 1-5 块标志**

for name, cnt in [('COCO 风格', counts_coco), ('TSR 风格', counts_tsr)]:
    print('%-10s mean=%.2f  p99=%d  max=%d' % (name, cnt.mean(),
                                               np.percentile(cnt, 99), cnt.max()))
print('\n%-6s %10s %12s %14s %16s %16s'
      % ('N', '覆盖率(TSR)', '覆盖率(COCO)', '正样本比例(TSR)', 'self-attn O(N^2)', '匹配 O(M^2 N)'))
rows = []
for N in [10, 30, 100, 300, 900]:
    cov_t = float((counts_tsr <= N).mean()); cov_c = float((counts_coco <= N).mean())
    pos_t = counts_tsr.mean() / N
    rows.append((N, cov_t, cov_c, pos_t))
    print('%-6d %10.5f %12.5f %14.4f %16.1e %16.1e'
          % (N, cov_t, cov_c, pos_t, N * N, (counts_coco.mean() ** 2) * N))

covs_t = [r[1] for r in rows]; poss = [r[3] for r in rows]
assert all(covs_t[i] <= covs_t[i + 1] for i in range(len(covs_t) - 1)), '覆盖率随 N 单调不减'
assert all(poss[i] > poss[i + 1] for i in range(len(poss) - 1)), '正样本比例随 N 单调下降'
assert rows[2][1] > 0.9999, 'TSR 场景 N=100 早已 100% 覆盖'

print('\n⚠️  常见误解：「DINO 用 900 个 query 是为了检更多目标」—— **不是**。')
print('    本合成分布单图最多 %d 个实例（真实 COCO 是 63），100 早就够了（覆盖率 %.5f）。'
      % (counts_coco.max(), rows[2][2]))
print('    提到 300/900 的真实动机是 **加密监督信号、加快收敛**，代价是正样本比例暴跌，')
print('    所以必须同时把 softmax CE 换成 sigmoid focal loss（模块 02 第 8 节）。')
print('\n✅ **TSR 场景的推理**：目标数分布 mean=%.2f、max=%d，N=100 覆盖率已是 %.5f。'
      % (counts_tsr.mean(), counts_tsr.max(), rows[2][1]))
print('   继续加到 900 不会提升召回，只会把正样本比例从 %.2f%% 压到 %.2f%%。'
      % (100 * rows[2][3], 100 * rows[4][3]))
print('   但也不该砍到 20：① DETR 论文的合成实验显示 query 的**实际可用容量只有名义值的一半**；')
print('   ② N 太小会加剧 query 之间的竞争与匹配不稳定；③ N=100 的 self-attn 开销本就可忽略。')
print('   → **N 取 50-100 是合理区间**（练习 3 把这个推理写成决策脚本）。')

## ✏️ 练习 1：Conditional DETR 的「内容项 + 空间项」分解

原始 DETR 把位置编码**相加**进 Q/K，展开后会多出两个交叉项
`c_q·p_k` 和 `p_q·c_k` —— 内容与位置互相污染。
Conditional DETR 改成只保留 `c_q·c_k + p_q·p_k`。

实现 `conditional_logits(cq, ck, pq, pk)`：返回 `(nq, nk)` 的 attention logits
`= (cq·ck + pq·pk) / sqrt(d_c + d_p)`。

In [ ]:
def conditional_logits(cq, ck, pq, pk):
    # TODO: 内容项与空间项各自点积后相加，再除以 sqrt(总维度)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
d16 = 16
cq = rng.normal(size=(6, d16)); ck = rng.normal(size=(20, d16))
pq = rng.normal(size=(6, d16)); pk = rng.normal(size=(20, d16))
L = conditional_logits(cq, ck, pq, pk)
assert L.shape == (6, 20)

# 性质 1：等价于把 [content; spatial] **拼接**后做一次点积
cat_q = np.concatenate([cq, pq], -1); cat_k = np.concatenate([ck, pk], -1)
assert np.allclose(L, cat_q @ cat_k.T / np.sqrt(2 * d16)), '分解 == 拼接后单次点积'

# 性质 2：原始 DETR 的「相加」会引入两个交叉项，且量级与正题相当（= 严重污染）
wanted = cq @ ck.T + pq @ pk.T
cross  = cq @ pk.T + pq @ ck.T
assert np.allclose((cq + pq) @ (ck + pk).T, wanted + cross)
ratio = float(np.abs(cross).mean() / np.abs(wanted).mean())
print('交叉项 / 正题 的平均幅值比 = %.2f  ← **交叉项一点都不小**' % ratio)
assert ratio > 0.5

# 性质 3：内容项置零时，空间项能把注意力精确拉到参考点上
grid = np.linspace(0, 1, 64)
pk_s = sinusoidal_pe(grid, d=128); refs = np.array([0.20, 0.50, 0.85])
pq_s = sinusoidal_pe(refs, d=128)
A_s = softmax(conditional_logits(np.zeros((3, 128)), np.zeros((64, 128)), pq_s, pk_s), -1)
peaks = grid[A_s.argmax(1)]
print('参考点 %s -> 注意力峰值位置 %s' % (refs, np.round(peaks, 3)))
assert np.abs(peaks - refs).max() < 0.05, '空间项应把注意力拉到参考点'
print('✅ 练习 1 通过：**把「看哪里」和「找什么」拆开**，是 Conditional DETR 收敛快 6.7x 的核心。')

## ✏️ 练习 2：DAB/DINO 的逐层框精修（iterative box refinement）

框被约束在 `[0,1]`，所以增量要加在 **inverse-sigmoid 空间**里再映射回来：
`b_{l+1} = sigmoid(inverse_sigmoid(b_l) + Δ_l)`。

实现 `inverse_sigmoid` 与 `apply_delta`。要求：**零增量必须是恒等变换**。

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def inverse_sigmoid(x, eps=1e-5):
    # TODO: sigmoid 的反函数；先把 x clip 到 [eps, 1-eps] 防 log(0)
    raise NotImplementedError

def apply_delta(box, delta):
    # TODO: 在 inverse-sigmoid 空间累加增量，再 sigmoid 回 [0,1]
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
def iou_cxcywh(a, b):
    ax1, ay1, ax2, ay2 = a[0]-a[2]/2, a[1]-a[3]/2, a[0]+a[2]/2, a[1]+a[3]/2
    bx1, by1, bx2, by2 = b[0]-b[2]/2, b[1]-b[3]/2, b[0]+b[2]/2, b[1]+b[3]/2
    iw = max(0.0, min(ax2, bx2) - max(ax1, bx1))
    ih = max(0.0, min(ay2, by2) - max(ay1, by1))
    I = iw * ih
    return I / (a[2] * a[3] + b[2] * b[3] - I + 1e-12)

z = np.array([-2.0, 0.0, 3.0])
assert np.allclose(inverse_sigmoid(sigmoid(z)), z, atol=1e-4)
box = np.array([0.40, 0.50, 0.20, 0.20])
assert np.allclose(apply_delta(box, np.zeros(4)), box, atol=1e-5), '零增量必须是恒等变换'

# 模拟 6 层 decoder 的逐层精修（每层把 logit 空间的残差消掉 60%）
gt = np.array([0.78, 0.30, 0.05, 0.05])      # 一块远处的小标志
b = box.copy(); ious = []
print('%-8s %28s %10s' % ('decoder 层', 'anchor (cx, cy, w, h)', 'IoU'))
for l in range(7):
    ious.append(iou_cxcywh(b, gt))
    print('%-8s %28s %10.4f'
          % ('anchor' if l == 0 else 'L%d' % l, np.round(b, 4), ious[-1]))
    if l < 6:
        b = apply_delta(b, 0.6 * (inverse_sigmoid(gt) - inverse_sigmoid(b)))

assert ious[0] == 0.0, '初始 anchor 与 GT 不相交 —— IoU 恒为 0（模块 02 的梯度死区！）'
assert all(ious[i] <= ious[i + 1] + 1e-12 for i in range(len(ious) - 1)), 'IoU 应单调不降'
assert ious[-1] > 0.85
print('\n✅ 练习 2 通过：**query 一旦被解释成框，就能被逐层精修** ——')
print('   这与模块 02 的辅助损失（每层都要输出可用的框）是天生一对，')
print('   也是 RT-DETR「同一份权重支持多档速度」的结构基础（跑前 k 层即可）。')
print('⚠️  注意前两层 IoU 恒为 0（框还不相交）—— 此时 GIoU 才是唯一有梯度的项。')

## ✏️ 练习 3：给定目标数分布，选一个 N

实现 `choose_num_queries(counts, candidates, target_coverage=0.9999)`：
- `rows`：每个候选 N 一条 `(N, coverage, pos_ratio, selfattn_cost)`
  （`coverage = P(M <= N)`，`pos_ratio = E[M]/N`，`selfattn_cost = N*N`）
- `recommend`：**满足覆盖率要求的最小 N**；若都不满足则取最大候选

In [ ]:
def choose_num_queries(counts, candidates, target_coverage=0.9999):
    # TODO: 返回 dict(rows=[...], recommend=N)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
CANDS = [10, 20, 50, 100, 300, 900]
r_tsr = choose_num_queries(counts_tsr, CANDS)
r_coco = choose_num_queries(counts_coco, CANDS)
print('%-6s %12s %14s %14s' % ('N', '覆盖率', '正样本比例', 'self-attn'))
for N, cov, pos, cost in r_tsr['rows']:
    print('%-6d %12.5f %14.4f %14.0f' % (N, cov, pos, cost))
print('\nTSR  分布推荐 N =', r_tsr['recommend'])
print('COCO 分布推荐 N =', r_coco['recommend'])

covs = [r[1] for r in r_tsr['rows']]
poss = [r[2] for r in r_tsr['rows']]
assert all(covs[i] <= covs[i + 1] for i in range(len(covs) - 1)), '覆盖率随 N 单调不减'
assert all(poss[i] > poss[i + 1] for i in range(len(poss) - 1)), '正样本比例随 N 单调下降'
assert r_tsr['recommend'] <= 20, 'TSR 分布下 N=20 已足够覆盖'
assert r_coco['recommend'] > r_tsr['recommend'], 'COCO 目标更多，需要更大的 N'
# 极端情况：候选都不够时退化为最大候选
assert choose_num_queries(counts_coco, [2, 3])['recommend'] == 3
print('\n✅ 练习 3 通过：**覆盖率只是下界**。实际取 N 还要考虑：')
print('   ① query 的实际可用容量只有名义值的一半（DETR 论文的 100 目标合成实验）；')
print('   ② N 太小加剧竞争与匹配不稳定；③ N 太大压低正样本比例、逼你换 focal loss。')
print('   → 工程上的合理做法：**从 p99.99 目标数出发，乘 2-4 倍安全系数，再向上取整到常用档位**。')

## ✏️ 练习 4：注意力的「有效关注范围」——DETR 收敛慢的量化诊断

定义 `effective_span(A) = exp(H(A))`，`H` 是每行注意力分布的熵。
它的含义是「这一行实际上在看多少个 key」：均匀分布 → `K`，one-hot → `1`。

实现 `attn_entropy(A)` 与 `effective_span(A)`（支持任意前置维度，最后一维是 key）。

In [ ]:
def attn_entropy(A):
    # TODO: 沿最后一维算熵，注意 log(0) 保护
    raise NotImplementedError

def effective_span(A):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
Kk = 850                                   # 800x1066 输入 / stride 32 的典型 key 数
A_uni = np.full((4, Kk), 1.0 / Kk)
assert np.allclose(effective_span(A_uni), Kk), '均匀分布的有效范围 = K'
A_one = np.zeros((4, Kk)); A_one[:, 3] = 1.0
assert np.allclose(effective_span(A_one), 1.0, atol=1e-6), 'one-hot 的有效范围 = 1'

logits = rng.normal(size=(4, Kk))
print('%-14s %16s %14s' % ('logits 缩放', '有效关注范围', '占全部 key'))
spans = []
for t in [0.0, 0.5, 1.0, 2.0, 4.0, 8.0]:
    s = float(effective_span(softmax(logits * t, -1)).mean())
    spans.append(s)
    print('%-14.1f %16.1f %13.1f%%' % (t, s, 100 * s / Kk))
assert spans[0] > 0.99 * Kk, 't=0（等价随机初始化）时应几乎均匀'
assert all(spans[i] > spans[i + 1] for i in range(len(spans) - 1)), '越尖越小'

# 用第 2 节真实跑出来的 cross-attention 量一下
span_real = float(effective_span(A_cross_demo.reshape(-1, A_cross_demo.shape[-1])).mean())
print('\n第 2 节 decoder 初始化时的 cross-attn 有效范围 = %.1f / %d 个 key（%.0f%%）'
      % (span_real, HW, 100 * span_real / HW))
assert span_real > 0.4 * HW
print('\n✅ 练习 4 通过：**这是 DETR 需要 500 epoch 的第一个根因的量化指标**。')
print('   初始 span ≈ K ≈ 1000 -> 每个 key 只分到千分之一的梯度；')
print('   模型必须先花掉大量 epoch 把注意力「收窄」，才谈得上学定位。')
print('⚠️  训练时把这个指标画出来：**如果它长期不下降，说明 cross-attention 没在聚焦**，')
print('   继续调学习率是没用的 —— 要换结构（可变形注意力 / 空间先验），见模块 04。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def conditional_logits(cq, ck, pq, pk):
    d_total = cq.shape[-1] + pq.shape[-1]
    return (cq @ ck.T + pq @ pk.T) / np.sqrt(d_total)

In [ ]:
# 练习 2 参考答案
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def inverse_sigmoid(x, eps=1e-5):
    x = np.clip(np.asarray(x, dtype=float), eps, 1.0 - eps)
    return np.log(x / (1.0 - x))

def apply_delta(box, delta):
    return sigmoid(inverse_sigmoid(box) + np.asarray(delta, dtype=float))

In [ ]:
# 练习 3 参考答案
def choose_num_queries(counts, candidates, target_coverage=0.9999):
    counts = np.asarray(counts)
    rows, rec = [], None
    for N in sorted(candidates):
        cov = float((counts <= N).mean())
        rows.append((N, cov, float(counts.mean() / N), float(N * N)))
        if rec is None and cov >= target_coverage:
            rec = N
    return {'rows': rows, 'recommend': rec if rec is not None else max(candidates)}

In [ ]:
# 练习 4 参考答案
def attn_entropy(A):
    A = np.asarray(A, dtype=float)
    return -(A * np.log(A + 1e-12)).sum(-1)

def effective_span(A):
    return np.exp(attn_entropy(A))

---
## 🧪 真实工程胶囊：DETR decoder 的配置、调试与常见 bug

In [ ]:
RECIPE = r'''
# ============ Object query / decoder：工程 checklist（PyTorch 伪代码 + 真实参数） ============

# ---- 1. query 的定义（DETR 官方 models/detr.py） ----
self.query_embed = nn.Embedding(num_queries, hidden_dim)     # 100 x 256，**可学习参数**
tgt = torch.zeros_like(query_embed)                          # **内容初始化为全 0**
# 注意：query_embed 与图像无关，在所有图上共享 —> 它编码的是「搜索策略」，不是「某个物体」

# ---- 2. decoder 一层（位置编码的加法规则，**记死**） ----
def forward(tgt, memory, pos, query_pos):
    q = k = tgt + query_pos                                  # self-attn: pos 进 Q/K
    tgt = tgt + self.self_attn(q, k, value=tgt)[0]           # **value 不加 pos**
    tgt = self.norm1(tgt)
    tgt = tgt + self.cross_attn(query=tgt + query_pos,       # cross-attn: pos 进 Q
                                key=memory + pos,            #             pos 进 K
                                value=memory)[0]             # **value 不加 pos**
    tgt = self.norm2(tgt)
    tgt = self.norm3(tgt + self.ffn(tgt))
    return tgt
# 规则：位置编码进 Q 和 K，**永远不进 V**。看到实现把 pos 加进 V，基本可判定是 bug。

# ---- 3. 典型超参 ----
NUM_QUERIES  = 100      # DETR;  300 (Deformable/DAB);  900 (DINO)
D_MODEL      = 256
NHEADS       = 8
DEC_LAYERS   = 6        # 每层都算辅助损失（模块 02）
PRE_NORM     = False    # DETR 用 post-LN；部分变体用 pre-LN（更稳但需调 lr）

# ---- 4. DAB-DETR 风格的 anchor query（把 query 显式写成 4D 框） ----
self.refpoint_embed = nn.Embedding(num_queries, 4)           # (x, y, w, h)，sigmoid 前
ref = self.refpoint_embed.weight.sigmoid()                   # -> [0,1]^4
query_pos = gen_sineembed_for_position(ref[..., :2])         # 位置编码由 (x,y) 生成
pos_x = pos_x * (ref_w_h[..., 0] / obj_w).unsqueeze(-1)      # **用 w/h 调制注意力窗口宽度**
# 逐层精修：
new_ref = (inverse_sigmoid(ref) + self.bbox_head[l](tgt)).sigmoid()
# **梯度要 detach 上一层的 ref**（不然会跨层耦合出不稳定），DINO 的 look-forward-twice 改了这点

# ---- 5. 四个必看的调试信号 ----
# [A] cross-attention 热力图: A[h, i].reshape(H, W) —— query 有没有找到目标？
#     TSR 里如果 query 关注的是杆件而不是牌面 -> 特征分辨率不够，该加 P2 / 多尺度
# [B] **重复率**: 同一 GT 附近出现 >=2 个高分预测的比例。长期不降 =>
#     query 太少 / 匹配不稳定 / self-attn 被错误 mask
# [C] **query 使用直方图**: 每个 query 被匹配的次数。出现大量「死 query」=>
#     N 偏大，或数据分布过窄（query 特化到了训练集的偏置上）
# [D] **effective span** = exp(H(attn)): 长期 ≈ K 说明 cross-attn 没在聚焦 -> 换结构

# ---- 6. 五个常见 bug ----
# [ ] 把 query_pos 加进了 value（位置污染内容）
# [ ] tgt 用随机初始化而不是 0（DETR 官方是 0；随机初始化会破坏「第一层无需 self-attn」的性质）
# [ ] 改了 num_queries 却没改推理端的 top-k -> 直接改变召回上限
# [ ] N 从 100 提到 900 却仍用 softmax CE + eos_coef -> 背景项失控，必须换 sigmoid focal
# [ ] 逐层精修时忘了 detach 上一层的 reference -> 训练发散
'''
print(RECIPE)
for key in ['nn.Embedding(num_queries, hidden_dim)', 'torch.zeros_like',
            'value=tgt', 'value=memory', 'refpoint_embed', 'inverse_sigmoid',
            'effective span', 'sigmoid focal']:
    assert key in RECIPE, key
print('✅ 配方覆盖：query 定义 / pos 加法规则 / 超参 / DAB anchor / 4 个调试信号 / 5 个常见 bug')

### 小结

- **query 不是特征，是可学习的位置嵌入**（`nn.Embedding(100, 256)`），与图像无关、跨图共享。
  内容 `tgt` 初始化为**全 0**，靠 cross-attention 逐层灌进图像证据。
  **位置编码只进 Q/K，永远不进 V。**
- **职责分工是理解 DETR 的钥匙**：cross-attention 是 image→query 的唯一通道（取证据），
  self-attention 是 query↔query 的唯一通道（协商去重）。
  本 notebook 用扰动实验精确证明了：**拿掉 self-attn，decoder 就是 N 条互不相干的流水线**
  （改动 query 5，其余 query 输出逐比特不变）。
- **「DETR 为什么不需要 NMS」的完整答案 = 压力 + 通道**：一对一匹配在训练期制造
  「重复必被惩罚」的压力（模块 02），self-attention 提供执行这个压力的通道（本模块）。
  只答前者会被追问穿。
- **query 的空间特化是自发涌现的，且是数据分布的镜像**。在 TSR 偏置分布上训练后，
  query 会集中到「地平线上方 + 道路右侧」；换到左行/城市道路，覆盖距离恶化 2 倍以上——
  **这是离线 mAP 看不出、实车成片掉召回的失效模式**。评测必须按画面位置分桶。
- **N 的选择**：必须 > 单图最大目标数，且实际可用容量只有名义值的一半。
  把 N 提到 300/900 的动机是**加密监督、加快收敛**，不是「检更多目标」；
  代价是正样本比例暴跌，必须同步换 focal loss。**TSR 场景 50–100 足够。**
- **DAB-DETR 的调制原理**：正弦位置编码的点积是只依赖距离的钟形核，
  把坐标除以 `w` 就精确地把核**拉宽 w 倍** —— 宽目标宽窗口，小标志窄窗口。
  query 一旦成为 4D 框，就能被逐层精修，与辅助损失天生一对。
- **query vs anchor**：功能同构（都是空间假设的载体），
  本质差别只有两条——**一对一 vs 一对多监督**（决定要不要 NMS），
  以及**query 之间能通信、anchor 不能**。
  「anchor-free vs anchor-based」已不是好的分类维度，**「一对一 vs 一对多」才是**。

下一站：**模块 04 · 收敛难题与 DETR 家族演进** —— 500 epoch 到底慢在哪，
Deformable / DN / DINO 各自砍掉了哪一刀。